In [15]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# ============================================
# USER INPUTS
# ============================================
TARGETS_CSV = "Targets.csv"
TARGET_NAME = "K2-100"   
NEIGHBOR_CSV = f"FRIENDS/{TARGET_NAME}.csv"

HOSTNAMES_TO_CHECK = [
    "K2-101",
    "K2-102",
    "K2-103",
    "K2-104",
    "EPIC_211901114",
    "K2-95",
]

# ============================================
# COLUMN DEFINITIONS
# ============================================
N_RA_COL = "RA"
N_DEC_COL = "DEC"
N_PLX_COL = "Plx(mas)"
POSSIBLE_NEIGHBOR_ID_COLS = ["GaiaDR3", "Source", "source_id", "DR3Name"]

POSSIBLE_NAME_COLS = ["hostname", "target", "Target", "pl_name", "name"]
T_RA_COL = "ra"
T_DEC_COL = "dec"
T_PLX_COL = "st_parallax_mas"
T_GAIA_ID_COL = "gaia_dr3_id"

# ============================================
# HELPERS
# ============================================
def sph_to_cart(ra_deg, dec_deg, plx_mas):
    d_pc = 1000.0 / plx_mas
    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)

    x = d_pc * np.cos(dec) * np.cos(ra)
    y = d_pc * np.cos(dec) * np.sin(ra)
    z = d_pc * np.sin(dec)
    return x, y, z, d_pc

def normalize_name(x):
    return str(x).strip().lower().replace("_", " ")

def parse_gaia_id(x):
    if pd.isna(x):
        return pd.NA
    try:
        return str(int(float(x)))
    except:
        return pd.NA

# ============================================
# LOAD TARGET
# ============================================
targets = pd.read_csv(TARGETS_CSV)

name_col = next((c for c in POSSIBLE_NAME_COLS if c in targets.columns), None)
if name_col is None:
    raise ValueError("No valid name column found")

targets["_norm"] = targets[name_col].map(normalize_name)
targets["_gid"] = targets[T_GAIA_ID_COL].map(parse_gaia_id)

target_row = targets[targets["_norm"] == normalize_name(TARGET_NAME)]

if len(target_row) == 0:
    raise ValueError("Target not found")

target_row = target_row.iloc[0]

tx, ty, tz, _ = sph_to_cart(
    float(target_row[T_RA_COL]),
    float(target_row[T_DEC_COL]),
    float(target_row[T_PLX_COL])
)

# ============================================
# LOAD NEIGHBORS
# ============================================
df = pd.read_csv(NEIGHBOR_CSV)

neighbor_id_col = next((c for c in POSSIBLE_NEIGHBOR_ID_COLS if c in df.columns), None)
if neighbor_id_col is None:
    raise ValueError("No Gaia ID column found in neighbor CSV")

df = df[(df[N_PLX_COL] > 0) & pd.notna(df[N_PLX_COL])].copy()
df["_gid"] = df[neighbor_id_col].map(parse_gaia_id)

x, y, z, d = sph_to_cart(
    df[N_RA_COL].values,
    df[N_DEC_COL].values,
    df[N_PLX_COL].values
)

df["x_pc"] = x - tx
df["y_pc"] = y - ty
df["z_pc"] = z - tz
df["r_pc"] = np.sqrt(df["x_pc"]**2 + df["y_pc"]**2 + df["z_pc"]**2)

# ============================================
# MATCH HOSTNAMES
# ============================================
check_names = [normalize_name(x) for x in HOSTNAMES_TO_CHECK]
check_rows = targets[targets["_norm"].isin(check_names)]

id_map = dict(zip(check_rows["_norm"], check_rows["_gid"]))

matched = df[df["_gid"].isin(id_map.values())].copy()

print("\n=== Matches inside this sphere ===")
if len(matched) == 0:
    print("None")
else:
    print(matched[["_gid", "r_pc"]].sort_values("r_pc").to_string(index=False))

# ============================================
# PLOT (FAST + COLORBAR RESTORED)
# ============================================
id_to_host = {v: k for k, v in host_to_id.items()}
matched["matched_hostname"] = matched["_gid"].map(id_to_host)

fig = go.Figure()

# Main cloud (WITH COLORBAR)
fig.add_trace(go.Scatter3d(
    x=df["x_pc"],
    y=df["y_pc"],
    z=df["z_pc"],
    mode="markers",
    marker=dict(
        size=2.5,
        color=df["r_pc"],
        colorscale="Viridis",
        opacity=0.85,
        showscale=True,
        colorbar=dict(title="r (pc)")
    ),
    name="Neighbors",
    hoverinfo="skip"
))

# Highlight matched hosts (RED)
if len(matched) > 0:
    fig.add_trace(go.Scatter3d(
        x=matched["x_pc"],
        y=matched["y_pc"],
        z=matched["z_pc"],
        mode="markers+text",                 
        marker=dict(
            size=6,
            color="red"
        ),
        text=matched["matched_hostname"],   
        textposition="top center",
        name="Matched hosts",
        hoverinfo="skip"
    ))

# Target (RED)
fig.add_trace(go.Scatter3d(
    x=[0],
    y=[0],
    z=[0],
    mode="markers+text",                 
    marker=dict(
        size=7,
        color="red"
    ),
    text=[TARGET_NAME],                  
    textposition="top center",
    name="Target",
    hoverinfo="skip"
))

fig.update_layout(
    title=f"3D Neighbor Distribution centered on {TARGET_NAME}",
    scene=dict(
        xaxis_title="X (pc)",
        yaxis_title="Y (pc)",
        zaxis_title="Z (pc)",
        aspectmode="data"
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()


=== Matches inside this sphere ===
              _gid     r_pc
661310756111835392 3.116340
659744295638254336 4.492472
664292459846946560 4.562888
661004400386393984 5.086160
